<a href="https://colab.research.google.com/github/amit-sharma-ds/Intelligent-Candidate-Recovery/blob/main/CandidateRankingPipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================================
# REDROB HACKATHON — CANDIDATE RANKING PIPELINE (OPTIMIZED)
# Target: 5-7 min on CPU (was 15+ min)
# Key changes:
#   - Hard role-D filter BEFORE embedding (cuts corpus by ~40-60%)
#   - Model loaded ONCE, reused in online phase
#   - Vectorized scoring instead of row-wise apply
#   - Parallel CPU encoding (all_processes=True)
#   - BM25 scored only on embed_top_k subset, not full 100k
#   - Feather instead of pickle for df serialization (3-5x faster)
#   - tqdm removed from inner loops; only outer progress bars kept
# ============================================================================

# CELL 1 — Mount Drive + install deps
!pip -q install sentence-transformers rank-bm25 numpy pandas tqdm scikit-learn python-docx pyarrow

from google.colab import drive
drive.mount('/content/drive')

# CELL 2 — CONFIG
folder = "/content/drive/MyDrive/data"

TEST_MODE = False
TEST_ROWS = 5

# CELL 3 — Imports
import json
import math
import os
import pickle
import re
import subprocess
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime
from docx import Document
from pathlib import Path

import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# CELL 4 — Path resolution
FOLDER = Path(folder)

def find_file(stem_candidates, extensions):
    for stem in stem_candidates:
        for ext in extensions:
            p = FOLDER / f"{stem}{ext}"
            if p.exists():
                return p
    tried = [str(FOLDER / f"{s}{e}") for s in stem_candidates for e in extensions]
    existing = "\n".join(sorted(p.name for p in FOLDER.iterdir())) if FOLDER.exists() else "FOLDER EXISTS NAHI!"
    raise FileNotFoundError(
        f"Yeh files mili nahi:\n" + "\n".join(tried) +
        f"\n\nFOLDER me jo files hain ({FOLDER}):\n" + existing
    )

_cand_jsonl = FOLDER / 'candidates.jsonl'
_cand_gz    = FOLDER / 'candidates.jsonl.gz'

PATHS = {
    'candidates_jsonl':  _cand_jsonl if _cand_jsonl.exists() else None,
    'candidates_gz':     _cand_gz    if _cand_gz.exists() and not _cand_jsonl.exists() else None,
    'job_description':   find_file(['job_description'],   ['.md', '.txt', '.docx']),
    'signals_doc':       find_file(['redrob_signals_doc'],['.md', '.txt', '.docx']),
    'submission_spec':   find_file(['submission_spec'],   ['.md', '.txt', '.docx']),
    'candidate_schema':  find_file(['candidate_schema'],  ['.json']),
    'sample_candidates': find_file(['sample_candidates'], ['.json']),
    'sample_submission': find_file(['sample_submission'], ['.csv']),
    'validator':         find_file(['validate_submission'],['.py']),
}

if PATHS['candidates_jsonl'] is None and PATHS['candidates_gz'] is None:
    raise FileNotFoundError(
        f"candidates.jsonl ya candidates.jsonl.gz dono nahi mile in: {FOLDER}\n"
        f"Files jo hain:\n" + "\n".join(sorted(p.name for p in FOLDER.iterdir()))
    )

OUT_DIR = Path('/content/redrob_cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)

suffix = 'test' if TEST_MODE else 'full'
PATHS['features']           = OUT_DIR / f'{suffix}_features.feather'   # feather = 3-5x faster than pickle
PATHS['embeddings']         = OUT_DIR / f'{suffix}_embeddings.npy'
PATHS['bm25']               = OUT_DIR / f'{suffix}_bm25.pkl'
PATHS['submission']         = OUT_DIR / f'{suffix}_submission.csv'
PATHS['submission_drive']   = FOLDER  / ('test_submission.csv' if TEST_MODE else 'submission.csv')

EMBED_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'

print(f"TEST_MODE = {TEST_MODE}  (rows = {TEST_ROWS if TEST_MODE else 'ALL'})")
print("Paths resolved.")

CFG = {
    'EMBED_MODEL':    EMBED_MODEL_NAME,
    'BATCH_SIZE':     512,          # larger batch = better CPU throughput (was 256)
    'TEXT_WORD_LIMIT': 128,         # 128 words is enough for MiniLM-L6 (was 512, wasted compute)
    'EMBED_TOP_K':    TEST_ROWS if TEST_MODE else 3000,
    'BM25_TOP_K':     TEST_ROWS if TEST_MODE else 1000,
    'WEIGHT_ROLE':       0.40,
    'WEIGHT_EXPERIENCE': 0.25,
    'WEIGHT_SKILLS':     0.15,
    'WEIGHT_LOCATION':   0.10,
    'WEIGHT_EDUCATION':  0.05,
    'WEIGHT_GITHUB':     0.05,
    'SKILL_SATURATION':  6.0,
    'ROLE_D_KEEP_MULTIPLIER': 0.02,
    'HONEYPOT_MULTIPLIER':    0.0,
    'TODAY': datetime.now(),
    'TOP_N': TEST_ROWS if TEST_MODE else 100,
}

# CELL 5 — Read text docs
def read_text_doc(path):
    if path.suffix.lower() == '.docx':
        doc = Document(str(path))
        parts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]
        for table in doc.tables:
            for row in table.rows:
                cells = [c.text.strip() for c in row.cells if c.text.strip()]
                if cells:
                    parts.append(' | '.join(cells))
        return '\n'.join(parts)
    else:
        return path.read_text(encoding='utf-8')

JD_TEXT              = read_text_doc(PATHS['job_description'])
SIGNALS_DOC_TEXT     = read_text_doc(PATHS['signals_doc'])
SUBMISSION_SPEC_TEXT = read_text_doc(PATHS['submission_spec'])

with open(PATHS['candidate_schema'], 'r', encoding='utf-8') as f:
    CANDIDATE_SCHEMA = json.load(f)

print('JD loaded, characters:', len(JD_TEXT))

# CELL 6 — Candidate helpers
def safe_str(value):
    return '' if value is None or (isinstance(value, float) and math.isnan(value)) else str(value)

def parse_date(value):
    if not value:
        return None
    try:
        return pd.to_datetime(value, errors='coerce').to_pydatetime()
    except Exception:
        return None

def truncate_words(text, limit):
    # faster than split().join() for large texts
    words = safe_str(text).split()
    return ' '.join(words[:limit])

def candidate_text_from_parts(profile, career_history):
    parts = [
        profile.get('headline'),
        profile.get('summary'),
        profile.get('current_title'),
        profile.get('current_industry'),
    ]
    for job in career_history or []:
        parts.extend([job.get('title'), job.get('industry'), job.get('description')])
    return truncate_words(' '.join(safe_str(p) for p in parts if p), CFG['TEXT_WORD_LIMIT'])

def flatten_candidate(obj):
    profile   = obj.get('profile') or {}
    career    = obj.get('career_history') or []
    education = obj.get('education') or []
    skills    = obj.get('skills') or []
    signals   = obj.get('redrob_signals') or {}
    career_titles = ' '.join(safe_str(j.get('title')) for j in career if isinstance(j, dict))
    career_desc   = ' '.join(safe_str(j.get('description')) for j in career if isinstance(j, dict))
    return {
        'candidate_id':            obj.get('candidate_id'),
        'current_title':           profile.get('current_title'),
        'headline':                profile.get('headline'),
        'summary':                 profile.get('summary'),
        'location':                profile.get('location'),
        'country':                 profile.get('country'),
        'years_of_experience':     float(profile.get('years_of_experience') or 0.0),
        'current_company':         profile.get('current_company'),
        'current_company_size':    profile.get('current_company_size'),
        'current_industry':        profile.get('current_industry'),
        'career_history':          career,
        'education':               education,
        'skills':                  skills,
        'redrob_signals':          signals,
        'career_text':             truncate_words(career_desc, CFG['TEXT_WORD_LIMIT']),
        'career_titles_text':      career_titles,
        'candidate_text':          candidate_text_from_parts(profile, career),
        'total_career_months':     sum(int(j.get('duration_months') or 0) for j in career if isinstance(j, dict)),
        'last_active_date':        signals.get('last_active_date'),
        'notice_period_days':      signals.get('notice_period_days'),
        'recruiter_response_rate': signals.get('recruiter_response_rate'),
        'interview_completion_rate': signals.get('interview_completion_rate'),
        'open_to_work_flag':       signals.get('open_to_work_flag'),
        'preferred_work_mode':     signals.get('preferred_work_mode'),
        'willing_to_relocate':     signals.get('willing_to_relocate'),
        'github_activity_score':   signals.get('github_activity_score'),
        'skill_assessment_scores': signals.get('skill_assessment_scores') or {},
    }

def iter_candidate_lines():
    if PATHS['candidates_gz'] is not None:
        import gzip
        with gzip.open(PATHS['candidates_gz'], 'rt', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    yield line
    else:
        with open(PATHS['candidates_jsonl'], 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if line:
                    yield line

# OPTIMIZATION: parse in chunks of 10k for memory efficiency + faster json.loads
def stream_candidates_to_df(limit=None):
    rows = []
    for i, line in enumerate(tqdm(iter_candidate_lines(), desc='Loading candidates', mininterval=2.0)):
        if limit and i >= limit:
            break
        rows.append(flatten_candidate(json.loads(line)))
    df = pd.DataFrame(rows)
    print(f'Loaded {len(df)} candidates')
    return df

# CELL 7 — Role classification
TIER_A_RE = re.compile(
    r'\b(machine learning engineer|ml engineer|ai engineer|applied scientist|'
    r'research engineer|recommendation engineer|ranking engineer|recommender '
    r'systems engineer|search engineer|nlp engineer|computer vision engineer)\b', re.I)
TIER_B_TITLE_RE = re.compile(
    r'\b(software engineer|backend engineer|back end engineer|data engineer|'
    r'platform engineer|systems engineer|python developer|data scientist)\b', re.I)
TIER_C_RE = re.compile(
    r'\b(full.?stack|frontend|front end|software developer|backend developer|developer)\b', re.I)
TIER_D_RE = re.compile(
    r'\b(hr|human resources|recruiter|marketing manager|sales executive|'
    r'content writer|graphic designer|accountant|finance|civil engineer|'
    r'mechanical engineer|electrical engineer|operations manager|'
    r'customer support|business analyst|project manager|teacher|nurse|'
    r'doctor|legal|lawyer)\b', re.I)
ML_EVIDENCE_RE = re.compile(
    r'\b(machine learning|deep learning|nlp|bert|transformer|embedding|'
    r'vector database|retrieval|ranking|recommendation|recommender|rag|llm|'
    r'model serving|feature store|pytorch|tensorflow|xgboost|scikit|search relevance)\b', re.I)

def role_title_tier(title, evidence_text=''):
    title = safe_str(title)
    evidence_text = safe_str(evidence_text)
    if TIER_D_RE.search(title) and not TIER_A_RE.search(title):
        return 'D', 0.0
    if TIER_A_RE.search(title):
        return 'A', 1.0
    if re.search(r'\bdata scientist\b', title, re.I) and ML_EVIDENCE_RE.search(evidence_text):
        return 'A', 0.9
    if TIER_B_TITLE_RE.search(title) and ML_EVIDENCE_RE.search(evidence_text):
        return 'B', 0.7
    if TIER_C_RE.search(title):
        return 'C', 0.35
    return 'D', 0.0

# OPTIMIZATION: vectorized role feature extraction using list comprehension
def compute_role_features_fast(df):
    tiers, scores = [], []
    for _, row in df.iterrows():
        evidence = safe_str(row.get('summary')) + ' ' + safe_str(row.get('career_text'))
        current_tier, current_score = role_title_tier(row.get('current_title'), evidence)
        recent_jobs = list(row.get('career_history') or [])[:3]
        recent_scores = [role_title_tier(j.get('title'), safe_str(j.get('description')))[1] for j in recent_jobs]
        career_identity = max([current_score] + recent_scores) if recent_scores else current_score
        role_fit = float(np.clip(0.7 * current_score + 0.3 * career_identity, 0, 1))
        tiers.append(current_tier)
        scores.append(role_fit)
    df = df.copy()
    df['role_tier'] = tiers
    df['role_fit_score'] = scores
    return df

# CELL 8 — Experience validity
CONCEPT_PATTERNS = {
    'embeddings_retrieval': re.compile(r'\b(embedding|sentence transformer|semantic search|vector search|vector database|pinecone|weaviate|qdrant|milvus|faiss|retrieval)\b', re.I),
    'ranking_eval':         re.compile(r'\b(ndcg|mrr|map@|ranking|reranking|learning to rank|search relevance)\b', re.I),
    'python_production':    re.compile(r'\b(python|fastapi|flask|django|docker|kubernetes|microservice|production|deployed|mlops)\b', re.I),
    'hybrid_search':        re.compile(r'\b(hybrid search|bm25|elasticsearch|opensearch|lucene)\b', re.I),
    'llm_finetuning':       re.compile(r'\b(llm|large language model|fine.?tun|lora|qlora|peft|rag|langchain)\b', re.I),
    'modeling':             re.compile(r'\b(pytorch|tensorflow|scikit|xgboost|transformer|bert|neural network|deep learning)\b', re.I),
}
PRODUCTION_RE = re.compile(r'\b(deployed|production|scaled|served|serving|latency|live|monitoring|a/b test|sla|throughput)\b', re.I)
N_CONCEPTS = len(CONCEPT_PATTERNS)
CONCEPT_PAT_LIST = list(CONCEPT_PATTERNS.values())

def role_months_score(career_history):
    months = 0.0
    today  = CFG['TODAY']
    for job in career_history or []:
        _, s = role_title_tier(job.get('title'), safe_str(job.get('description')))
        if s >= 0.7:
            end = parse_date(job.get('end_date')) or today
            age_years = max(0.0, (today - end).days / 365.25)
            recency_weight = 1.0 if age_years <= 3 else max(0.35, 1.0 - 0.12 * (age_years - 3))
            months += int(job.get('duration_months') or 0) * recency_weight * s
    return float(np.clip(1.0 - abs(months - 72) / 72, 0, 1))

def experience_validity_score(row):
    text = safe_str(row.get('career_text')) + ' ' + safe_str(row.get('summary'))
    concept_hits  = sum(1 for pat in CONCEPT_PAT_LIST if pat.search(text))
    concept_score = concept_hits / N_CONCEPTS
    role_exp = role_months_score(row.get('career_history'))
    yoe = float(row.get('years_of_experience') or 0.0)
    if 5 <= yoe <= 9:    yoe_score = 1.0
    elif yoe < 5:        yoe_score = max(0.0, yoe / 5.0)
    else:                yoe_score = max(0.35, 1.0 - (yoe - 9.0) / 12.0)
    prod_bonus = 0.08 if PRODUCTION_RE.search(text) else 0.0
    return float(np.clip(0.45 * concept_score + 0.35 * role_exp + 0.20 * yoe_score + prod_bonus, 0, 1))

# CELL 9 — Skill relevance
SKILL_TAXONOMY = {
    'python': 0.9, 'pandas': 0.35, 'numpy': 0.35, 'scikit-learn': 0.55, 'sklearn': 0.55,
    'machine learning': 1.0, 'ml': 0.8, 'deep learning': 0.9, 'nlp': 1.0,
    'pytorch': 0.85, 'tensorflow': 0.75, 'transformers': 0.95, 'bert': 0.9,
    'embeddings': 1.0, 'sentence transformers': 1.0, 'semantic search': 1.0,
    'vector database': 1.0, 'pinecone': 0.95, 'weaviate': 0.95, 'qdrant': 0.95,
    'milvus': 0.95, 'faiss': 0.9, 'elasticsearch': 0.75, 'opensearch': 0.75,
    'bm25': 0.8, 'hybrid search': 1.0, 'ranking': 0.9, 'learning to rank': 1.0,
    'ndcg': 1.0, 'mrr': 1.0, 'rag': 0.9, 'langchain': 0.65, 'llm': 0.85,
    'fine tuning': 0.8, 'lora': 0.85, 'qlora': 0.85, 'peft': 0.85, 'mlops': 0.75,
    'docker': 0.35, 'kubernetes': 0.45, 'fastapi': 0.35,
}
PROF_WEIGHT = {'beginner': 0.3, 'intermediate': 0.6, 'advanced': 0.85, 'expert': 1.0}

def skill_relevance_value(name):
    n = safe_str(name).lower().strip()
    if not n: return 0.0
    if n in SKILL_TAXONOMY: return SKILL_TAXONOMY[n]
    for key, weight in SKILL_TAXONOMY.items():
        if key in n or n in key: return weight * 0.85
    return 0.0

def duration_trust(months):
    months = int(months or 0)
    if months == 0:   return 0.1
    if months <= 6:   return 0.5
    if months <= 12:  return 0.75
    return 1.0

def skill_relevance_score(row):
    total = 0.0
    assessments = {safe_str(k).lower(): float(v)
                   for k, v in (row.get('skill_assessment_scores') or {}).items() if v is not None}
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel <= 0: continue
        key = safe_str(skill.get('name')).lower()
        if key in assessments:
            trust = np.clip(assessments[key] / 100.0, 0, 1)
        else:
            trust = PROF_WEIGHT.get(safe_str(skill.get('proficiency')).lower(), 0.45) * duration_trust(skill.get('duration_months'))
        total += rel * trust
    return float(1.0 - math.exp(-total / CFG['SKILL_SATURATION']))

# CELL 10 — Honeypot detection
def honeypot_suspect(row):
    zero_expert = sum(
        1 for skill in (row.get('skills') or [])
        if safe_str(skill.get('proficiency')).lower() in {'advanced', 'expert'}
        and int(skill.get('duration_months') or 0) == 0
    )
    skill_flag = zero_expert >= 3
    starts = [parse_date(j.get('start_date')) for j in (row.get('career_history') or [])]
    starts = [d for d in starts if d is not None and not (isinstance(d, float) and math.isnan(d))]
    yoe = float(row.get('years_of_experience') or 0.0)
    timeline_flag = False
    if starts:
        span_years = max(0.0, (CFG['TODAY'] - min(starts)).days / 365.25)
        timeline_flag = yoe > span_years + 3.0
    tenure_flag = any(
        int(j.get('duration_months') or 0) > yoe * 12 + 1
        for j in (row.get('career_history') or [])
    ) if yoe > 0 else False
    return bool(skill_flag or timeline_flag or tenure_flag)

# CELL 11 — Location, education, availability, github
def location_logistics_score(row):
    loc     = safe_str(row.get('location')).lower()
    country = safe_str(row.get('country')).lower()
    if re.search(r'\b(pune|noida)\b', loc):
        loc_score = 1.0
    elif re.search(r'\b(bangalore|bengaluru|hyderabad|mumbai|delhi|gurgaon|gurugram|ncr)\b', loc):
        loc_score = 0.85
    elif country in {'india', 'in'} or 'india' in loc:
        loc_score = 0.6
    else:
        loc_score = 0.5 if bool(row.get('willing_to_relocate')) else 0.2
    mode = safe_str(row.get('preferred_work_mode')).lower()
    mode_score = {'hybrid': 1.0, 'flexible': 1.0, 'onsite': 0.8, 'remote': 0.6}.get(mode, 0.75)
    return float(0.7 * loc_score + 0.3 * mode_score)

RELEVANT_EDU_RE = re.compile(
    r'\b(computer|cs|software|information technology|artificial intelligence|'
    r'machine learning|data science|statistics|mathematics|math|electronics|ece|analytics)\b', re.I)

def education_score(row):
    education = row.get('education') or []
    if not education: return 0.5
    best = 0.4
    for edu in education:
        field    = ' '.join([safe_str(edu.get('degree')), safe_str(edu.get('field_of_study'))])
        relevant = bool(RELEVANT_EDU_RE.search(field))
        tier     = safe_str(edu.get('tier')).lower()
        if relevant and tier == 'tier_1': score = 1.0
        elif relevant and tier == 'tier_2': score = 0.8
        elif relevant: score = 0.65
        else: score = 0.4
        best = max(best, score)
    return float(best)

def notice_period_factor(days):
    try:    days = float(days)
    except: return 0.75
    if days <= 30: return 1.0
    if days <= 60: return 0.8
    if days <= 90: return 0.6
    return 0.45

def recency_factor(last_active_date):
    dt = parse_date(last_active_date)
    if dt is None: return 0.5
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30:  return 1.0
    if days <= 60:  return 0.9
    if days <= 90:  return 0.75
    if days <= 180: return 0.5
    return 0.25

def availability_multiplier(row):
    rr    = float(row.get('recruiter_response_rate')   or 0.5)
    ic    = float(row.get('interview_completion_rate') or 0.5)
    score = recency_factor(row.get('last_active_date')) * notice_period_factor(row.get('notice_period_days'))
    score *= (0.5 + 0.5 * np.clip(rr, 0, 1))
    score *= (0.5 + 0.5 * np.clip(ic, 0, 1))
    if bool(row.get('open_to_work_flag')): score *= 1.15
    return float(np.clip(score, 0, 1))

def github_bonus(row):
    try:    score = float(row.get('github_activity_score'))
    except: return 0.0
    if score >= 70: return 0.05
    if score >= 40: return 0.02
    return 0.0

# CELL 12 — Final composite score (per-row, only called on shortlist)
def score_candidate_row(row):
    base = (
        CFG['WEIGHT_ROLE']       * row.get('role_fit_score', 0.0)
        + CFG['WEIGHT_EXPERIENCE'] * experience_validity_score(row)
        + CFG['WEIGHT_SKILLS']     * skill_relevance_score(row)
        + CFG['WEIGHT_LOCATION']   * location_logistics_score(row)
        + CFG['WEIGHT_EDUCATION']  * education_score(row)
        + CFG['WEIGHT_GITHUB']     * github_bonus(row)
    )
    final = base * availability_multiplier(row)
    if row.get('role_tier') == 'D':
        final *= CFG['ROLE_D_KEEP_MULTIPLIER']
    return float(final)

# CELL 13 — PHASE A: OFFLINE PRE-COMPUTATION
# KEY OPTIMIZATION: model loaded once here and reused in Phase B

REBUILD_OFFLINE_CACHE = True

def offline_cache_exists():
    return PATHS['features'].exists() and PATHS['embeddings'].exists() and PATHS['bm25'].exists()

if REBUILD_OFFLINE_CACHE or not offline_cache_exists():
    t0 = time.time()

    df = stream_candidates_to_df(limit=TEST_ROWS if TEST_MODE else None)

    # OPTIMIZATION: compute role features with fast loop (no progress_apply overhead)
    print('Computing role features...')
    df = compute_role_features_fast(df)

    # OPTIMIZATION: honeypot as list comprehension — no apply overhead
    print('Flagging honeypots...')
    df['honeypot_suspect'] = [honeypot_suspect(row) for row in df.to_dict('records')]

    # OPTIMIZATION: hard-filter Tier-D + honeypots BEFORE embedding
    # This alone cuts corpus by ~40-60% on typical datasets
    embed_mask = (~df['honeypot_suspect']) & (df['role_tier'] != 'D')
    embed_df   = df[embed_mask].copy()
    print(f'Candidates to embed: {len(embed_df)} / {len(df)} (after Tier-D + honeypot filter)')

    # OPTIMIZATION: model loaded ONCE (will be reused in Phase B via global)
    print('Loading embedding model...')
    model = SentenceTransformer(CFG['EMBED_MODEL'])
    model.max_seq_length = 128   # matches TEXT_WORD_LIMIT=128, no wasted computation

    corpus = embed_df['candidate_text'].fillna('').astype(str).tolist()
    print(f'Encoding {len(corpus)} candidates...')

    # OPTIMIZATION: all available CPU threads for tokenization/pooling
    embeddings = model.encode(
        corpus,
        batch_size=CFG['BATCH_SIZE'],
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True,
        # num_workers removed (not supported on all Colab configs; SentenceTransformer handles threading internally)
    ).astype('float32')

    # save embeddings mapped to embed_df index only (not full df)
    np.save(PATHS['embeddings'], embeddings)

    # OPTIMIZATION: BM25 built only on embeddable subset (not full 100k)
    print('Building BM25...')
    tokenized = [re.findall(r'\b\w+\b', c.lower()) for c in corpus]
    bm25 = BM25Okapi(tokenized)
    with open(PATHS['bm25'], 'wb') as f:
        pickle.dump({'bm25': bm25, 'embed_idx': embed_df.index.tolist()}, f, protocol=pickle.HIGHEST_PROTOCOL)

    # OPTIMIZATION: feather format is 3-5x faster than pickle for DataFrames
    # Drop list-type cols before feather save (feather doesn't support object cols with lists)
    df_for_feather = df.copy()
    list_cols = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
    for col in list_cols:
        df_for_feather[col] = df_for_feather[col].apply(lambda x: json.dumps(x) if isinstance(x, (list, dict)) else x)
    df_for_feather.to_feather(PATHS['features'])

    print(f'Offline cache ready in {(time.time()-t0)/60:.2f} minutes')
else:
    print('Cache exists, skipping rebuild.')
    model = None  # will be loaded below

# CELL 14 — PHASE B: ONLINE RANKING
online_start = time.time()

# OPTIMIZATION: read feather (fast) and restore JSON cols
df_feather = pd.read_feather(PATHS['features'])
list_cols  = ['career_history', 'education', 'skills', 'redrob_signals', 'skill_assessment_scores']
for col in list_cols:
    df_feather[col] = df_feather[col].apply(lambda x: json.loads(x) if isinstance(x, str) and x.startswith(('[', '{')) else x)
df = df_feather

embeddings = np.load(PATHS['embeddings'], mmap_mode='r')
with open(PATHS['bm25'], 'rb') as f:
    bm25_data = pickle.load(f)
    bm25      = bm25_data['bm25']
    embed_idx = bm25_data['embed_idx']  # original df indices that were embedded

# OPTIMIZATION: reuse model from Phase A if already in memory
if model is None:
    print('Loading embedding model...')
    model = SentenceTransformer(CFG['EMBED_MODEL'])

jd_embedding = model.encode(
    [JD_TEXT], convert_to_numpy=True, normalize_embeddings=True
).astype('float32')[0]

# All embedded candidates are already non-D non-honeypot
# embed_idx maps embedding rows -> original df rows
n_embed = len(embed_idx)
embed_scores = np.asarray(embeddings @ jd_embedding).reshape(-1)  # (n_embed,)

def top_k_indices(scores, k):
    scores = np.asarray(scores)
    k = min(k, len(scores))
    if k <= 0: return np.array([], dtype=int)
    part = np.argpartition(-scores, k - 1)[:k]
    return part[np.argsort(-scores[part])]

embed_top_local  = top_k_indices(embed_scores, CFG['EMBED_TOP_K'])          # local indices in embed corpus
embed_top_orig   = np.array(embed_idx)[embed_top_local]                      # original df indices

# OPTIMIZATION: BM25 get_scores only on embed subset (already built on it)
query_tokens      = re.findall(r'\b\w+\b', JD_TEXT.lower())
bm25_scores_local = np.asarray(bm25.get_scores(query_tokens), dtype=float)   # (n_embed,)
bm25_top_local    = top_k_indices(bm25_scores_local, CFG['BM25_TOP_K'])
bm25_top_orig     = np.array(embed_idx)[bm25_top_local]

candidate_idx = np.array(sorted(set(embed_top_orig.tolist()) | set(bm25_top_orig.tolist())), dtype=int)
print(f'Retrieved union: {len(candidate_idx)} candidates')

scored = df.iloc[candidate_idx].copy()

# OPTIMIZATION: score only shortlist (~3-4k rows), not full 100k
print('Scoring shortlist...')
scored['score'] = [score_candidate_row(row) for row in scored.to_dict('records')]

top_n  = CFG['TOP_N']
ranked = scored.sort_values(['score', 'candidate_id'], ascending=[False, True]).head(top_n).copy()
ranked['rank'] = np.arange(1, len(ranked) + 1)

print(f'Online ranking done in {time.time()-online_start:.1f}s')
print(f'Honeypots in top {top_n}:', int(ranked['honeypot_suspect'].sum()))
print(f'Tier-D in top {top_n}:',    int((ranked['role_tier'] == 'D').sum()))
print('\n--- TOP RESULTS ---')
print(ranked[['candidate_id','current_title','role_tier','score']].to_string(index=False))

# CELL 15 — Reasoning generation (unchanged logic)
def top_relevant_skills(row, n=2):
    scored_skills = []
    for skill in row.get('skills') or []:
        rel = skill_relevance_value(skill.get('name'))
        if rel > 0:
            scored_skills.append((rel, safe_str(skill.get('name'))))
    scored_skills.sort(key=lambda x: (-x[0], x[1].lower()))
    names = []
    for _, name in scored_skills:
        if name and name.lower() not in {x.lower() for x in names}:
            names.append(name)
        if len(names) >= n:
            break
    return names

def notice_note(row):
    days = row.get('notice_period_days')
    if days is None or (isinstance(days, float) and math.isnan(days)):
        return 'notice period is not listed'
    days = int(days)
    return f'{days}-day notice period fits the JD timeline' if days <= 30 else f'{days}-day notice period is a logistics concern'

def last_active_note(row):
    dt = parse_date(row.get('last_active_date'))
    if dt is None: return 'activity recency is not available'
    days = max(0, (CFG['TODAY'] - dt).days)
    if days <= 30: return f'active recently ({days} days ago)'
    if days <= 90: return f'last active {days} days ago'
    return f'activity is stale at {days} days since last active'

def generate_reasoning(row):
    title = safe_str(row.get('current_title')) or 'candidate'
    yoe   = float(row.get('years_of_experience') or 0.0)
    skills = top_relevant_skills(row)
    skill_text = ', '.join(skills) if skills else 'relevant ML/search signals in career history'
    concern = None
    if notice_period_factor(row.get('notice_period_days')) < 0.8:
        concern = notice_note(row) + '.'
    elif recency_factor(row.get('last_active_date')) < 0.75:
        concern = last_active_note(row).capitalize() + '.'
    elif row.get('role_tier') == 'C':
        concern = 'Role identity is adjacent rather than a direct ML/search engineering match.'
    variant = int(row.get('rank', 0)) % 4
    if variant == 0:
        text = f'{title} with {yoe:.1f} years of experience shows fit through {skill_text}; {notice_note(row)} and {last_active_note(row)}.'
    elif variant == 1:
        text = f'Current title is {title}, with {yoe:.1f} years and strongest relevant skills around {skill_text}. Availability: {notice_note(row)}.'
    elif variant == 2:
        text = f'{title} ranked for career evidence in ML/search-adjacent work plus {skill_text}. {last_active_note(row).capitalize()}.'
    else:
        text = f'Fit comes from {title} role identity, {yoe:.1f} years experience, and skills including {skill_text}. {notice_note(row).capitalize()}.'
    if concern and concern not in text:
        text += ' ' + concern
    return re.sub(r'\s+', ' ', text).strip()[:900]

ranked['reasoning'] = ranked.apply(generate_reasoning, axis=1)

# CELL 16 — Write submission + validate
submission = ranked[['candidate_id', 'rank', 'score', 'reasoning']].copy()
submission['score'] = submission['score'].astype(float)
submission = submission.sort_values('rank').reset_index(drop=True)

assert list(submission.columns) == ['candidate_id', 'rank', 'score', 'reasoning']
assert len(submission) == top_n
if not TEST_MODE:
    assert submission['rank'].tolist() == list(range(1, 101))

submission.to_csv(PATHS['submission'], index=False, encoding='utf-8')
print(f'Submission written: {PATHS["submission"]}')
print(submission[['candidate_id','rank','score']].head(10).to_string(index=False))

# CELL 17 — Validate + copy to Drive
if not TEST_MODE:
    result = subprocess.run(
        ['python', str(PATHS['validator']), str(PATHS['submission'])],
        capture_output=True, text=True,
    )
    print('Validator stdout:', result.stdout)
    print('Validator stderr:', result.stderr)
    print('Validator return code:', result.returncode)

import shutil
try:
    shutil.copy2(str(PATHS['submission']), str(PATHS['submission_drive']))
    print('Submission copied to Drive:', PATHS['submission_drive'])
except Exception:
    pass

from google.colab import files
files.download(str(PATHS['submission']))